<a href="https://colab.research.google.com/github/Sam3885/Ai-car-lease-assistant/blob/main/CAR_LEASE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving SakshamThakur_contract.pdf to SakshamThakur_contract (1).pdf


In [3]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.1 MB/s eta 0:00:00


In [5]:
import PyPDF2
file_name = list(uploaded.keys())[0]
with open(file_name, 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    text = ""
    for page in reader.pages:
        text += page.extract_text()

print(text[:1000])

 
A67_ 0523  Page 1 of 7  
 
  
LESSOR  
(“us”, “we”, “our”)  Legal Name:  
 Trading Name:  
 Address:  
 
  
    
LESSEE  
(“you”)  Name   TAX INVOICE   
Address   GST NO. 49 -809-999  
 Email Address   LEASE NO.    
   
 
SPECIFIC INFORMATION  
Motor Vehicle  MAKE  
 MODEL  
 VIN NO.  
 REG. NO.  
 YEAR  
 ODOMETER  
      (kms/hrs) 
ACCESSORIES  
 
Insurance Details  INSURER  
 POLICY NO.  
 
 
DISCLOSURE STATEMENT for your consumer credit contract  
IMPORTANT  – We are required to provide you with this Disclosure Statement under section 17 of the Credit Contracts and Consumer Finance Act 
2003. This Disclosure Statement sets out the key information about this contract. You should read it thoroughly. If you do no t understand anything 
in this document, you should seek independent advice. You should keep this contract in a safe place.  
This Disclosure Statement must be provided to you before  this contract is entered into by you.  The law gives you a limited right to c ancel this c

In [8]:
sla_fields = {
    "apr_percent": None,
    "term_months": None,
    "monthly_payment": None,
    "down_payment": None,
    "residual_value": None,
    "mileage_allowance": None,
    "mileage_overage_fee": None,
    "early_termination_fee": None,
    "purchase_option_price": None,
    "insurance_requirements": None,
    "maintenance_responsibilities": None,
    "warranty_summary": None,
    "late_fee_policy": None
}
print(text[:1000])

 
A67_ 0523  Page 1 of 7  
 
  
LESSOR  
(“us”, “we”, “our”)  Legal Name:  
 Trading Name:  
 Address:  
 
  
    
LESSEE  
(“you”)  Name   TAX INVOICE   
Address   GST NO. 49 -809-999  
 Email Address   LEASE NO.    
   
 
SPECIFIC INFORMATION  
Motor Vehicle  MAKE  
 MODEL  
 VIN NO.  
 REG. NO.  
 YEAR  
 ODOMETER  
      (kms/hrs) 
ACCESSORIES  
 
Insurance Details  INSURER  
 POLICY NO.  
 
 
DISCLOSURE STATEMENT for your consumer credit contract  
IMPORTANT  – We are required to provide you with this Disclosure Statement under section 17 of the Credit Contracts and Consumer Finance Act 
2003. This Disclosure Statement sets out the key information about this contract. You should read it thoroughly. If you do no t understand anything 
in this document, you should seek independent advice. You should keep this contract in a safe place.  
This Disclosure Statement must be provided to you before  this contract is entered into by you.  The law gives you a limited right to c ancel this c

In [10]:
import re
apr_match = re.search(r"(APR|Annual Percentage Rate|Interest Rate)[^\d]*([\d\.]+)", text, re.IGNORECASE)
if apr_match:
    sla_fields["apr_percent"] = apr_match.group(2)
print("Extracted APR:", sla_fields["apr_percent"])

Extracted APR: 365.


In [11]:
snippet = re.search(r"(APR|Interest Rate|Annual.*Rate).{0,50}", text, re.IGNORECASE)
print(snippet.group(0) if snippet else "Not found")

Annual Interest Rate            % fixed for the whole term of this con


In [12]:
apr_match = re.search(r"([\d\.]+)\s*percent", text, re.IGNORECASE)
if apr_match:
    sla_fields["apr_percent"] = apr_match.group(1)
print("Extracted APR:", sla_fields["apr_percent"])

Extracted APR: 365.


In [13]:
apr_match = re.search(r"([\d\.]+)\s*(%|percent)", text, re.IGNORECASE)
if apr_match:
    sla_fields["apr_percent"] = apr_match.group(1)
print("Extracted APR:", sla_fields["apr_percent"])

Extracted APR: 5.00


In [15]:
# Monthly Payment
monthly_match = re.search(r"(Monthly Payment|Payment Amount)[^\d]*\$?([\d,\.]+)", text, re.IGNORECASE)
if monthly_match:
    sla_fields["monthly_payment"] = monthly_match.group(2)

# Term (months)
term_match = re.search(r"(\d+)\s*(months|month term)", text, re.IGNORECASE)
if term_match:
    sla_fields["term_months"] = term_match.group(1)

# Down Payment
down_match = re.search(r"(Down Payment|Initial Payment)[^\d]*\$?([\d,\.]+)", text, re.IGNORECASE)
if down_match:
    sla_fields["down_payment"] = down_match.group(2)

# Residual Value
residual_match = re.search(r"(Residual Value|End Value)[^\d]*\$?([\d,\.]+)", text, re.IGNORECASE)
if residual_match:
    sla_fields["residual_value"] = residual_match.group(2)

# Mileage Allowance
mileage_match = re.search(r"(Mileage Allowance|Annual Mileage)[^\d]*(\d+)", text, re.IGNORECASE)
if mileage_match:
    sla_fields["mileage_allowance"] = mileage_match.group(2)

# Mileage Overage Fee
overage_match = re.search(r"(Overage Fee|Excess Mileage)[^\d]*\$?([\d,\.]+)", text, re.IGNORECASE)
if overage_match:
    sla_fields["mileage_overage_fee"] = overage_match.group(2)

In [16]:
import json

with open("contract_sla.json", "w") as f:
    json.dump(sla_fields, f, indent=4)

print("Saved SLA fields to contract_sla.json")

Saved SLA fields to contract_sla.json


In [18]:
import requests

vin = "1HGCM82633A004352"  # sample VIN
url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVin/{vin}?format=json"
response = requests.get(url)
data = response.json()

# Initialize vehicle_info dictionary
vehicle_info = {
    "make": None,
    "model": None,
    "year": None
}

# Extract useful fields by iterating through the results
results = data["Results"]
for item in results:
    if item["Variable"] == "Make":
        vehicle_info["make"] = item["Value"]
    elif item["Variable"] == "Model":
        vehicle_info["model"] = item["Value"]
    elif item["Variable"] == "Model Year":
        vehicle_info["year"] = item["Value"]

print(vehicle_info)

{'make': 'HONDA', 'model': 'Accord', 'year': '2003'}


In [19]:
import json

contract_data = {
    "sla_fields": sla_fields,   # your extracted SLA fields
    "vehicle_info": vehicle_info  # the VIN decoded info
}

with open("contract_with_vehicle.json", "w") as f:
    json.dump(contract_data, f, indent=4)

print("Saved combined contract + vehicle data")

Saved combined contract + vehicle data


In [20]:
explanations = {
    "apr_percent": "APR (Annual Percentage Rate) is the yearly interest charged on the loan.",
    "term_months": "Term is the length of the loan or lease in months.",
    "monthly_payment": "Monthly Payment is the fixed amount you pay each month.",
    "down_payment": "Down Payment is the upfront amount you pay at the start of the contract.",
    "residual_value": "Residual Value is the car’s estimated worth at the end of the lease.",
    "mileage_allowance": "Mileage Allowance is the maximum miles you can drive per year without penalty.",
    "mileage_overage_fee": "Mileage Overage Fee is the charge per mile if you exceed the allowance.",
    "early_termination_fee": "Early Termination Fee is the penalty if you end the contract early.",
    "purchase_option_price": "Purchase Option Price is the amount you pay if you want to buy the car at the end of the lease.",
    "insurance_requirements": "Insurance Requirements specify what coverage you must maintain.",
    "maintenance_responsibilities": "Maintenance Responsibilities explain who handles servicing and repairs.",
    "warranty_summary": "Warranty Summary describes what parts or services are covered by warranty.",
    "late_fee_policy": "Late Fee Policy is the penalty if you miss or delay a payment."
}

for field, value in sla_fields.items():
    if value:
        print(f"{field}: {value} → {explanations.get(field, 'No explanation available')}")

apr_percent: 5.00 → APR (Annual Percentage Rate) is the yearly interest charged on the loan.
residual_value: 365. → Residual Value is the car’s estimated worth at the end of the lease.


In [22]:
negotiation_points = []

# APR check
if sla_fields.get("apr_percent"):
    apr = float(sla_fields["apr_percent"])
    if apr > 7:
        negotiation_points.append(f"APR {apr}% is high compared to typical rates (3–7%). Consider negotiating lower.")
    else:
        negotiation_points.append(f"APR {apr}% is reasonable compared to typical rates (3–7%).")

# Term check
if sla_fields.get("term_months"):
    term = int(sla_fields["term_months"])
    if term > 60:
        negotiation_points.append(f"Term {term} months is long. Shorter terms reduce total interest paid.")
    elif term < 12:
        negotiation_points.append(f"Term {term} months is very short. This reduces interest but increases monthly payments.")
    else:
        negotiation_points.append(f"Term {term} months is within a standard range (12–60).")

# Mileage allowance check
if sla_fields.get("mileage_allowance"):
    mileage = int(sla_fields["mileage_allowance"])
    if mileage < 12000:
        negotiation_points.append(f"Mileage allowance {mileage} is low. Standard is 12,000–15,000 miles/year.")
    else:
        negotiation_points.append(f"Mileage allowance {mileage} is good compared to the standard 12,000–15,000 miles/year.")

print("Negotiation Suggestions:")
for point in negotiation_points:
    print("-", point)

Negotiation Suggestions:
- APR 5.0% is reasonable compared to typical rates (3–7%).


In [24]:
import pandas as pd
import json

# Example SLA fields (replace with your extracted values)
sla_fields = {
    "apr_percent": "5",
    "term_months": "6",
    "monthly_payment": "15000",
    "down_payment": "50000",
    "mileage_allowance": "10000"
}

# Example vehicle info (replace with your VIN lookup results)
vehicle_info = {
    "make": "HONDA",
    "model": "Accord",
    "year": "2003"
}

# Negotiation suggestions (from Step 16)
negotiation_points = [
    "APR 5% is reasonable compared to typical rates (3–7%).",
    "Term 6 months is very short. This reduces interest but increases monthly payments.",
    "Mileage allowance 10000 is low. Standard is 12,000–15,000 miles/year."
]

# ---- Presentation Layer ----

print("=== Contract SLA Fields ===")
sla_df = pd.DataFrame(list(sla_fields.items()), columns=["Field", "Value"])
print(sla_df.to_string(index=False))

print("\n=== Vehicle Information ===")
vehicle_df = pd.DataFrame(list(vehicle_info.items()), columns=["Attribute", "Value"])
print(vehicle_df.to_string(index=False))

print("\n=== Negotiation Suggestions ===")
for point in negotiation_points:
    print("•", point)

# Optional: Save everything neatly into one JSON file
contract_data = {
    "sla_fields": sla_fields,
    "vehicle_info": vehicle_info,
    "negotiation_suggestions": negotiation_points
}

with open("final_contract_demo.json", "w") as f:
    json.dump(contract_data, f, indent=4)

print("\n✅ Final demo data saved to final_contract_demo.json")

=== Contract SLA Fields ===
            Field Value
      apr_percent     5
      term_months     6
  monthly_payment 15000
     down_payment 50000
mileage_allowance 10000

=== Vehicle Information ===
Attribute  Value
     make  HONDA
    model Accord
     year   2003

=== Negotiation Suggestions ===
• APR 5% is reasonable compared to typical rates (3–7%).
• Term 6 months is very short. This reduces interest but increases monthly payments.
• Mileage allowance 10000 is low. Standard is 12,000–15,000 miles/year.

✅ Final demo data saved to final_contract_demo.json
